# 07-2. requests 기초 응답 처리 예제

## Goal

- 응답 크기를 제한해 읽습니다.
- 상태 코드와 Content-Type을 함께 검증합니다.

이 노트북은 교안 예제를 안전하게 재현하는 보조 실습입니다. 먼저 결과를 예측한 뒤 셀을 실행하세요.


## Setup

가짜 응답 객체를 사용하므로 `requests` 설치와 네트워크가 필요하지 않습니다.


## Steps

### 제한된 응답 읽기

실제 `requests.Response.iter_content()`와 같은 인터페이스를 가진 객체로 처리 경계를 연습합니다.


In [1]:
class FakeResponse:
    def __init__(self, status_code, headers, chunks):
        self.status_code = status_code
        self.headers = headers
        self._chunks = chunks

    def iter_content(self, chunk_size):
        yield from self._chunks


def read_limited(response, limit=1024):
    parts = []
    size = 0
    for chunk in response.iter_content(128):
        if not chunk:
            continue
        size += len(chunk)
        if size > limit:
            raise ValueError("응답 크기 제한을 초과했습니다")
        parts.append(chunk)
    return b"".join(parts)


response = FakeResponse(200, {"Content-Type": "application/json; charset=utf-8"}, [b'{"status":', b' "ok"}'])
body = read_limited(response)
media_type = response.headers["Content-Type"].split(";", 1)[0].strip().lower()
print(response.status_code, media_type, body.decode("utf-8"))


200 application/json {"status": "ok"}


## Checks

정상 응답과 크기 초과 응답을 확인합니다.


In [2]:
assert response.status_code == 200
assert media_type == "application/json"
try:
    read_limited(FakeResponse(200, {}, [b"x" * 1025]))
except ValueError:
    print("과도한 응답 거부 확인")


과도한 응답 거부 확인


## Next Steps

실제 요청에서는 연결 타임아웃과 읽기 타임아웃을 튜플로 명시하고 리다이렉트 정책을 정합니다.
